Test a 100x100 or more random symbolic expression matrix and multiply it using numpy objects, see if this is the slowdown. Maybe the reason is the heyoka c++ to python interface being called repeatedly, where it should only be called once?

In [2]:
import numpy as np
import heyoka as hy
import time

n = 240
rng = np.random.default_rng(42)

# Create symbolic Heyoka variables
x, y = hy.make_vars("x", "y")

# Generate random coefficients
a = rng.standard_normal((n, n))
b = rng.standard_normal((n, n))

# Construct matrices containing symbolic Heyoka expressions
A = np.array(
    [[float(value) * x for value in row] for row in a],
    dtype=object
)

B = np.array(
    [[float(value) * y for value in row] for row in b],
    dtype=object
)

# Benchmark symbolic matrix multiplication
start = time.perf_counter()
C = A @ B
elapsed = time.perf_counter() - start

print(f"Heyoka matrix multiplication: {elapsed:.6f} seconds")
print(f"Result shape: {C.shape}")
print(f"Result entry type: {type(C[0, 0])}")

# Compare against ordinary NumPy multiplication
start = time.perf_counter()
C_numeric = a @ b
elapsed_numeric = time.perf_counter() - start

print(f"NumPy float multiplication: {elapsed_numeric:.6f} seconds")

Heyoka matrix multiplication: 72.516485 seconds
Result shape: (240, 240)
Result entry type: <class 'heyoka._core.expression'>
NumPy float multiplication: 0.044153 seconds


In [3]:
import numpy as np
import heyoka as hy
import time
import gc

n = 100
rng = np.random.default_rng(42)

a = rng.random((n, n))
b = rng.random((n, n))

x, y = hy.make_vars("x", "y")

# 1. Ordinary numerical matrices
A_float = a
B_float = b

# 2. Python object matrices holding numeric floats
A_object = a.astype(object)
B_object = b.astype(object)

# 3. Matrices of Heyoka constant expressions
A_const = np.array([
    [hy.expression(float(v)) for v in row]
    for row in a
], dtype=object)

B_const = np.array([
    [hy.expression(float(v)) for v in row]
    for row in b
], dtype=object)

# 4. Matrices of genuinely symbolic Heyoka expressions
A_symbolic = np.array([
    [float(v) * x for v in row]
    for row in a
], dtype=object)

B_symbolic = np.array([
    [float(v) * y for v in row]
    for row in b
], dtype=object)


def benchmark(name, A, B):
    gc.collect()

    start = time.perf_counter()
    C = A @ B
    duration = time.perf_counter() - start

    print(f"{name:27s}: {duration:.6f} s")
    return duration


print("100x100 matrix multiplication\n")

t_float = benchmark("NumPy float64", A_float, B_float)
t_object = benchmark("Python float objects", A_object, B_object)
t_const = benchmark("Heyoka constants", A_const, B_const)
t_symbolic = benchmark("Heyoka symbolic", A_symbolic, B_symbolic)

print("\nRelative slowdown:")
print(f"Object / float64:       {t_object / t_float:.1f}x")
print(f"Heyoka / Python object: {t_symbolic / t_object:.1f}x")
print(f"Symbolic / constant:    {t_symbolic / t_const:.1f}x")

100x100 matrix multiplication

NumPy float64              : 0.000677 s
Python float objects       : 0.059636 s
Heyoka constants           : 3.409974 s
Heyoka symbolic            : 5.056667 s

Relative slowdown:
Object / float64:       88.1x
Heyoka / Python object: 84.8x
Symbolic / constant:    1.5x


In [4]:
def heyoka_matmul_sum(A, B):
    """Matrix multiplication using Heyoka's multivariate sums."""
    rows, inner = A.shape
    inner_b, cols = B.shape

    if inner != inner_b:
        raise ValueError("Incompatible dimensions")

    C = np.empty((rows, cols), dtype=object)

    for i in range(rows):
        for j in range(cols):
            terms = [
                A[i, k] * B[k, j]
                for k in range(inner)
            ]
            C[i, j] = hy.sum(terms)

    return C

In [7]:
# Start smaller to keep the experiment manageable
N = 240

A_test = A_symbolic[:N, :N]
B_test = B_symbolic[:N, :N]

start = time.perf_counter()
C_numpy = A_test @ B_test
t_numpy = time.perf_counter() - start

start = time.perf_counter()
C_heyoka = heyoka_matmul_sum(A_test, B_test)
t_heyoka = time.perf_counter() - start

print(f"NumPy object matmul: {t_numpy:.6f} s")
print(f"Heyoka sum matmul:   {t_heyoka:.6f} s")
print(f"Speedup:             {t_numpy / t_heyoka:.2f}x")

NumPy object matmul: 5.999523 s
Heyoka sum matmul:   3.446143 s
Speedup:             1.74x
